In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RadarData"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

# Region = "PRECIP"; Case = "WET"; spinup_hours = "12" 
Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12" 

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarData_PRECIP_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#LOADING RADAR CLASS
from scipy.spatial import Delaunay
from scipy.interpolate import LinearNDInterpolator

folderDirectory = os.path.join(
    DirectoryManager.dataDirectory,
    "Observation_Data/PRECIP/Radar",
    ModelData_NSSL.case
)

RadarData_PRECIP = RadarData_PRECIP_Class(ModelData_NSSL, folderDirectory)

In [ ]:
########################
#DATA INFORMATION

In [ ]:
#DATA CITATION

# PRECIP (https://www.eol.ucar.edu/field_projects/precip)
# Prediction of Rainfall Extremes Campaign in the Pacific

# PROJECT DATES
# 05/25/2022 - 08/10/2022
# Project Location
# Taiwan

# SPOL Radar https://www.eol.ucar.edu/observing_facilities/s-pol

In [ ]:
########################
#FUNCTIONS

In [ ]:
def CreateMask(ModelData):
    lonMin = np.nanmin(RadarData_PRECIP.longitude)
    lonMax = np.nanmax(RadarData_PRECIP.longitude)
    latMin = np.nanmin(RadarData_PRECIP.latitude)
    latMax = np.nanmax(RadarData_PRECIP.latitude)
    
    model_data = ModelData.GetDataTimestep_diag(t=0, varName="refl10cm")
    
    RadarDataMask = (
        (model_data.latitude >= latMin) &
        (model_data.latitude <= latMax) &
        (model_data.longitude >= lonMin) &
        (model_data.longitude <= lonMax)
    )
    
    return RadarDataMask

In [ ]:
def SaveMaskData(ModelData, RadarDataMask):
    codeType = os.path.join("DataAnalysis", "Observation_Data")
    dataType = "RadarData/RadarObservationMask"
    outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
    outputPath = os.path.join(outputDirectory,f"{ModelData.region}_{ModelData.case}_spinup{ModelData.spinup_hours}hrs")
    os.makedirs(outputPath, exist_ok=True)
    outputFileName = f"RadarObservationMask.nc"
    outputFileNamePath = os.path.join(outputPath, outputFileName)

    RadarDataMask.to_netcdf(outputFileNamePath)
    print(f"Saved to {outputFileNamePath}","\n")

In [ ]:
########################
#CREATING MASK

In [ ]:
# Creating Mask
RadarDataMask = CreateMask(ModelData_NSSL)

#Saving Mask
SaveMaskData(ModelData_NSSL, RadarDataMask)